# 🎯 Task 1A: Binary Persuasion Detection in Arabic
**Objective:** Determine whether an Arabic text snippet (news paragraph or tweet) contains persuasion techniques (`'true'`) or not (`'false'`).

### Pipeline Steps:
1. **Environment Setup & Path Configuration**
2. **Data Loading & Preprocessing** (Text cleaning + Outlier filtering)
3. **Fine-Tuning MARBERTv2** (`UBC-NLP/MARBERTv2`)
4. **Fine-Tuning AraBERTv02** (`aubmindlab/bert-base-arabertv02`)
5. **Validation Comparison & Model Selection**
6. **Generating Final Test Predictions & Exporting Submission**

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Ensure src/ is on sys.path so modules are imported directly from local files
NOTEBOOK_DIR = Path.cwd()
SRC_DIR = NOTEBOOK_DIR.parents[1] if 'Notebooks' in str(NOTEBOOK_DIR) else NOTEBOOK_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(), override=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

# Direct imports from our custom package
from decetction_persuation.preprocessing import preprocess_dataset
from decetction_persuation.train_marbert import train_marbert, predict_marbert
from decetction_persuation.train_arabert import train_arabert, predict_arabert

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))


c:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Version: 2.6.0+cu124
CUDA Available: True
GPU Device: NVIDIA GeForce GTX 1660 Ti


In [2]:
# Resolve data directory
DIR = NOTEBOOK_DIR.parents[2] / 'data' if 'Notebooks' in str(NOTEBOOK_DIR) else Path(r'C:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\data')

# Load raw Task 1A files
df_1a_train = pd.read_json(DIR / "task1A_train.jsonl", lines=True, encoding="utf-8")
df_1a_dev   = pd.read_json(DIR / "task1A_dev.jsonl", lines=True, encoding="utf-8")
df_1a_test  = pd.read_json(DIR / "task1A_test_no_label.jsonl", lines=True, encoding="utf-8")

print(f"Loaded Train: {len(df_1a_train):,} samples")
print(f"Loaded Dev:   {len(df_1a_dev):,} samples")
print(f"Loaded Test:  {len(df_1a_test):,} samples")
display(df_1a_train.head())


Loaded Train: 2,427 samples
Loaded Dev:   259 samples
Loaded Test:  503 samples


,id,text,label,type
0,00149,جوجل يحتفل بعيد استقلال المملكة الأردنية الهاش...,false,tweet
1,00022,🔴 الأمن #العراقي: الهجوم تسبب باستشهاد آمر الل...,true,tweet
2,FAT_336-eurl_02_011,وبحسب المنظمة الحقوقية الدولية، فقد دعت النياب...,false,paragraph
3,MIS_1069-curl_01_010,وما فتئت المنظمات الحقوقية الدولية توجّه المرة...,true,paragraph
4,00491,#وليد_جنبلاط لـmtv: فلننظر الى الواقع الذي نحن...,true,tweet


In [3]:
# Preprocess datasets (Arabic cleaning + length outlier removal)
df_train_clean = preprocess_dataset(
    df_1a_train,
    is_train=True,
    remove_outliers=True,
    max_words=100
)
df_dev_clean = preprocess_dataset(df_1a_dev, is_train=False)
df_test_clean = preprocess_dataset(df_1a_test, is_train=False)

print("\n--- Task 1A Label Distribution in Train ---")
print(df_train_clean['label'].value_counts(normalize=True) * 100)


[Pipeline Complete] Initial: 2,427 -> Final: 2,427 samples.
[Pipeline Complete] Initial: 259 -> Final: 259 samples.
[Pipeline Complete] Initial: 503 -> Final: 503 samples.

--- Task 1A Label Distribution in Train ---
label
true     79.027606
false    20.972394
Name: proportion, dtype: float64


In [4]:
# 1. Fine-Tune MARBERTv2 (Excels on Arabic tweets and dialectal content)
results_marbert = train_marbert(
    train_df=df_train_clean,
    val_df=df_dev_clean,
    task="task1A",
    model_name="UBC-NLP/MARBERTv2",
    epochs=4,
    batch_size=16,
    learning_rate=2e-5,
    max_len=128,
    warmup_ratio=0.1,
    weight_decay=0.01,
    loss_type="auto",
    fp16=True,
    early_stopping_patience=2,
    output_dir="./saved_models/marbert_task1a",
    seed=42,
)



🚀 [MARBERT] Training on device: cuda | Task: TASK1A | Model: UBC-NLP/MARBERTv2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11030.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture

Epoch 01 (299.3s) | Train Loss: 0.4826 | Val Loss: 0.4019 | F1_BINARY: 0.9007


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.55s/it]


  ⭐ New best f1_binary: 0.9007 -> Saved to marbert_task1a


...

Epoch 3/4 [Train]:   0%|          | 0/152 [00:00<?, ?it/s]c:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\src\decetction_persuation\train_marbert.py:176: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(fp16 and device == "cuda")):
Epoch 3/4 [Train]: 100%|██████████| 152/152 [04:44<00:00,  1.87s/it, loss=0.1326]
c:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\src\decetction_persuation\train_marbert.py:205: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(fp16 and device == "cuda")):


Epoch 03 (294.2s) | Train Loss: 0.2083 | Val Loss: 0.4796 | F1_BINARY: 0.9023


Epoch 4/4 [Train]:   0%|          | 0/152 [00:00<?, ?it/s]c:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\src\decetction_persuation\train_marbert.py:176: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(fp16 and device == "cuda")):
Epoch 4/4 [Train]: 100%|██████████| 152/152 [04:44<00:00,  1.87s/it, loss=0.0212]
c:\Users\pc\Desktop\M2 BDIO\LLMs\Decetction-Persuation\src\decetction_persuation\train_marbert.py:205: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(fp16 and device == "cuda")):


Epoch 04 (294.1s) | Train Loss: 0.1151 | Val Loss: 0.5739 | F1_BINARY: 0.8983
  🛑 Early stopping triggered after 4 epochs.


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 541.88it/s]


In [ ]:
# 2. Fine-Tune AraBERTv02 (Excels on formal Modern Standard Arabic news paragraphs)
results_arabert = train_arabert(
    train_df=df_train_clean,
    val_df=df_dev_clean,
    task="task1A",
    model_name="aubmindlab/bert-base-arabertv02",
    epochs=4,
    batch_size=16,
    learning_rate=2e-5,
    max_len=128,
    warmup_ratio=0.1,
    weight_decay=0.01,
    loss_type="auto",
    fp16=True,
    early_stopping_patience=2,
    output_dir="./saved_models/arabert_task1a",
    seed=42,
)


In [ ]:
# 3. Model Comparison & Leader Selection
print("=================== TASK 1A VALIDATION COMPARISON ===================")
print(f"MARBERT Validation F1: {results_marbert['best_metric']:.4f}")
print(f"AraBERT Validation F1: {results_arabert['best_metric']:.4f}")

if results_marbert['best_metric'] >= results_arabert['best_metric']:
    best_model = results_marbert['model']
    best_tokenizer = results_marbert['tokenizer']
    best_predictor = predict_marbert
    best_name = "MARBERTv2"
else:
    best_model = results_arabert['model']
    best_tokenizer = results_arabert['tokenizer']
    best_predictor = predict_arabert
    best_name = "AraBERTv02"

print(f"\n🏆 Best Model Selected for Final Predictions: {best_name}")


In [ ]:
# 4. Generate Final Predictions on Test Set
test_preds = best_predictor(
    model=best_model,
    tokenizer=best_tokenizer,
    texts=df_test_clean['text'].tolist(),
    task="task1A",
)

# Format output dataframe into competition submission standard (string 'true' / 'false')
df_submission = pd.DataFrame({
    "id": df_test_clean["id"],
    "label": ["true" if p == 1 else "false" for p in test_preds]
})

output_submission_path = DIR / "task1A_test_predictions.jsonl"
df_submission.to_json(output_submission_path, orient="records", lines=True, force_ascii=False)

print(f"✅ Successfully exported {len(df_submission):,} predictions to: {output_submission_path.name}")
display(df_submission.head(10))
